# Capítulo 10 - Exercícios Resolvidos (Worksheets)

**Quantum Computing for the Quantum Curious - Hughes et al.**

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np

---
## 10.1 - Correlation in Entangled States Lab
---

### Exercício 1
**Envie pares de partículas no estado produto |↑_A⟩|↓_B⟩. O que Alice e Bob medem?**

In [ ]:
# Estado produto: |↑⟩|↓⟩ = |0⟩|1⟩ = |01⟩
qc_product = QuantumCircuit(2)
qc_product.x(1)  # Qubit B em |1⟩ (spin down)
# Qubit A permanece em |0⟩ (spin up)

state = Statevector.from_instruction(qc_product)
print(f"Estado: {state}")
print("\nRESPOSTA:")
print("  Alice SEMPRE mede spin up (↑)")
print("  Bob SEMPRE mede spin down (↓)")
print("\n→ Estado produto: cada partícula tem estado DEFINIDO e INDEPENDENTE")

### Exercício 2
**Envie pares no estado emaranhado (1/√2)(|↑_A⟩|↓_B⟩ − |↓_A⟩|↑_B⟩). O que medem?**

In [ ]:
# Estado emaranhado (singleto): (|01⟩ - |10⟩)/√2
qc_entangled = QuantumCircuit(2, 2)
qc_entangled.x(1)
qc_entangled.h(0)
qc_entangled.cx(0, 1)
qc_entangled.z(0)  # Para criar o estado |Ψ-⟩

state = Statevector.from_instruction(qc_entangled.remove_final_measurements(inplace=False))
print(f"Estado: {state}")

# Simular medições
qc_entangled.measure([0, 1], [0, 1])
simulator = AerSimulator()
result = simulator.run(qc_entangled, shots=1000).result()
counts = result.get_counts()

print(f"\nResultados: {counts}")
print("\nRESPOSTA:")
print("  Cada um vê resultados aparentemente ALEATÓRIOS:")
print("    Alice: ~50% ↑, ~50% ↓")
print("    Bob: ~50% ↑, ~50% ↓")
print("\n  PORÉM, os resultados são PERFEITAMENTE ANTICORRELACIONADOS:")
print("    Quando Alice mede ↑, Bob SEMPRE mede ↓")
print("    Quando Alice mede ↓, Bob SEMPRE mede ↑")

### Exercício 3
**Se Alice mede seu spin, você conseguiria prever o resultado de Bob?**

**(a) No estado produto?**

**(b) No estado emaranhado?**

In [ ]:
print("""
RESPOSTAS:

(a) ESTADO PRODUTO |↑⟩|↓⟩:
    SIM, podemos prever!
    Bob SEMPRE mede ↓, independentemente do que Alice faz.
    O resultado de Bob é PREDETERMINADO.

(b) ESTADO EMARANHADO:
    SIM, podemos prever!
    Toda vez que Alice mede ↑, Bob mede ↓
    Toda vez que Alice mede ↓, Bob mede ↑
    
    A correlação é PERFEITA, mesmo que os resultados 
    individuais pareçam aleatórios!
""")

### Exercício 9 - Distinguindo Fontes

**Como distinguir entre:**
- Fonte #1: Emite aleatoriamente |↑↓⟩ ou |↓↑⟩ (mistura clássica)
- Fonte #2: Emite o estado emaranhado (|↑↓⟩ − |↓↑⟩)/√2

In [ ]:
print("""
COMO DISTINGUIR:

Na BASE Z: Ambas as fontes dão resultados anticorrelacionados!
          → NÃO conseguimos distinguir

SOLUÇÃO: Medir na BASE X!

┌─────────────────┬───────────────────────────────┐
│ Fonte           │ Resultado na base X           │
├─────────────────┼───────────────────────────────┤
│ #1 (mistura)    │ SEM correlação                │
│ #2 (emaranhado) │ Anticorrelação PERFEITA       │
└─────────────────┴───────────────────────────────┘

Se Alice e Bob SEMPRE obtêm resultados opostos na base X:
→ A fonte emite partículas EMARANHADAS

Se NÃO há correlação na base X:
→ As partículas NÃO estão emaranhadas (mistura clássica)
""")

---
## 10.4 - Schrödinger's Worm (IBM Q)
---

### Exercício 1-2: Estados Clássicos do Worm

In [ ]:
print("""
ESTADOS CLÁSSICOS:

Worm VIVO (5 quadrados pretos):  |11111⟩
  Todos os 5 bits são 1 (preto)

Worm MORTO (4 pretos, 1 branco): |11110⟩
  O bit mais à direita (q[0]) é 0 (branco)
  
Worm MUITO MORTO (destruído):    |11010⟩
  Padrão específico representando worm despedaçado
""")

### Exercício 3: Worm em Superposição (Vivo + Morto)

In [ ]:
# Criar worm em superposição de vivo e morto
qc_worm = QuantumCircuit(5, 5)

# Preparar estado base (vivo)
qc_worm.x([1, 2, 3, 4])  # q[1] até q[4] em |1⟩
qc_worm.x(0)  # q[0] em |1⟩ (vivo completo)

# Criar superposição em q[0]
qc_worm.h(0)  # Agora q[0] está em superposição

print("Circuito - Worm em Superposição:")
print(qc_worm.draw())

# Estado resultante
state = Statevector.from_instruction(qc_worm)
print("\nEstado: (1/√2)(|11110⟩ + |11111⟩)")
print("→ Superposição de MORTO e VIVO!")

In [ ]:
# Simular medições
qc_worm.measure(range(5), range(5))
result = simulator.run(qc_worm, shots=1000).result()
counts = result.get_counts()

print("Resultados da medição:")
for state, count in sorted(counts.items()):
    status = "VIVO" if state == "11111" else "MORTO"
    print(f"  |{state}⟩ ({status}): {count/10:.1f}%")

print("\n→ O colapso na medição é ALEATÓRIO (~50% cada)")

### Exercício 5: Trazer o Worm de Volta à Vida

In [ ]:
print("""
COMO TRAZER DE VOLTA À VIDA:

Aplicar outro HADAMARD em q[0]!

Todas as portas quânticas são REVERSÍVEIS.
O Hadamard é sua própria inversa: H† = H

Sequência:
  |11111⟩ (vivo)
     ↓ H
  (1/√2)(|11110⟩ + |11111⟩) (superposição)
     ↓ H
  |11111⟩ (vivo novamente!)

A segunda porta H DESFAZ a superposição, retornando ao estado original.
""")

# Demonstração
qc_revive = QuantumCircuit(5)
qc_revive.x([0, 1, 2, 3, 4])  # Worm vivo
qc_revive.h(0)  # Superposição
qc_revive.h(0)  # Desfaz superposição

state = Statevector.from_instruction(qc_revive)
print(f"\nEstado final: {state}")
print("→ Worm está VIVO novamente!")

### Exercício 8: Emaranhamento - Vivo vs Muito Morto

In [ ]:
# Superposição entre |11111⟩ (vivo) e |11010⟩ (muito morto)
# Diferenças: q[0] e q[2] mudam JUNTOS

qc_entangled_worm = QuantumCircuit(5, 5)

# Estado base
qc_entangled_worm.x([0, 1, 2, 3, 4])  # Todos em |1⟩

# Criar superposição em q[0]
qc_entangled_worm.h(0)

# EMARANHAR q[0] e q[2]: quando q[0] flipa, q[2] também flipa
qc_entangled_worm.cx(0, 2)

print("Circuito - Worm EMARANHADO (vivo ↔ muito morto):")
print(qc_entangled_worm.draw())

# Simular
qc_entangled_worm.measure(range(5), range(5))
result = simulator.run(qc_entangled_worm, shots=1000).result()
counts = result.get_counts()

print("\nResultados:")
for state, count in sorted(counts.items()):
    if state == "11111":
        status = "VIVO"
    elif state == "11010":
        status = "MUITO MORTO"
    else:
        status = "INESPERADO"
    print(f"  |{state}⟩ ({status}): {count/10:.1f}%")

print("\n→ q[0] e q[2] estão EMARANHADOS: mudam JUNTOS!")

---
## 10.5 - Superposition vs Mixed States Lab
---

### Exercício 1-2: Distinguindo Superposição de Mistura

In [ ]:
print("""
SUPERPOSIÇÃO vs MISTURA CLÁSSICA:

┌───────────────┬──────────────────┬──────────────────┐
│ Medição       │ Mistura Clássica │ Superposição     │
├───────────────┼──────────────────┼──────────────────┤
│ Base Z        │ 50% ↑, 50% ↓     │ 50% ↑, 50% ↓     │
│ Base X        │ 50% +, 50% −     │ 100% + (ou −)    │
└───────────────┴──────────────────┴──────────────────┘

SEMELHANÇA: Resultados IDÊNTICOS na base Z

DIFERENÇA: A superposição |+⟩ = (|0⟩+|1⟩)/√2 dá resultado 
           DETERMINÍSTICO na base X!
           A mistura continua 50/50.
""")

In [ ]:
# Demonstração matemática
print("""
EXPLICAÇÃO MATEMÁTICA:

SUPERPOSIÇÃO |+⟩ = (|0⟩ + |1⟩)/√2

Mudança de base:
  |0⟩ = (|+⟩ + |−⟩)/√2
  |1⟩ = (|+⟩ − |−⟩)/√2

Substituindo:
  |+⟩ = (1/√2)[(|+⟩ + |−⟩)/√2 + (|+⟩ − |−⟩)/√2]
      = (1/√2) × (2|+⟩)/√2
      = |+⟩

→ Medição na base X SEMPRE dá +x (100%)

════════════════════════════════════════════════════

MISTURA CLÁSSICA: 50% |0⟩ + 50% |1⟩ (NÃO é superposição!)

  50% em |0⟩ = (|+⟩ + |−⟩)/√2 → 50% +x, 50% −x
  50% em |1⟩ = (|+⟩ − |−⟩)/√2 → 50% +x, 50% −x
  
  Total: 50% +x, 50% −x (SEM interferência!)
""")

### Exercício 6-7: Encontrando α e β

In [ ]:
# Dado: Na base Z, medimos 20% spin up, 80% spin down
print("""
ENCONTRANDO α e β:

Estado: |ψ⟩ = α|0⟩ + β|1⟩

Das probabilidades na base Z:
  P(0) = α² = 0.2 → α = 1/√5 ≈ 0.447
  P(1) = β² = 0.8 → β = 2/√5 ≈ 0.894

Portanto: |ψ⟩ = (1/√5)|0⟩ + (2/√5)|1⟩
""")

# Verificação
alpha = 1/np.sqrt(5)
beta = 2/np.sqrt(5)

print(f"Verificação:")
print(f"  α² = {alpha**2:.3f} (esperado: 0.2)")
print(f"  β² = {beta**2:.3f} (esperado: 0.8)")
print(f"  α² + β² = {alpha**2 + beta**2:.3f} (deve ser 1.0)")

In [ ]:
# Verificar probabilidades na base X
print("""
VERIFICAÇÃO NA BASE X:

|ψ⟩ = (1/√5)|0⟩ + (2/√5)|1⟩

Substituindo |0⟩ e |1⟩:
  = (1/√5)[(|+⟩ + |−⟩)/√2] + (2/√5)[(|+⟩ − |−⟩)/√2]
  = (1/√10)(|+⟩ + |−⟩) + (2/√10)(|+⟩ − |−⟩)
  = (1/√10 + 2/√10)|+⟩ + (1/√10 − 2/√10)|−⟩
  = (3/√10)|+⟩ − (1/√10)|−⟩

Probabilidades:
  P(+x) = (3/√10)² = 9/10 = 90%
  P(−x) = (1/√10)² = 1/10 = 10%
""")

# Verificação numérica
coef_plus = 3/np.sqrt(10)
coef_minus = 1/np.sqrt(10)
print(f"Verificação numérica:")
print(f"  P(+x) = {coef_plus**2:.1%}")
print(f"  P(−x) = {coef_minus**2:.1%}")

---
## 10.6 - Measurement Basis Lab
---

In [ ]:
# Probabilidade de passar pelo SGA em função do ângulo
print("""
FÓRMULA: P(passa) = cos²(θ/2)

Ou equivalentemente: P(passa) = (1 + cos θ)/2
""")

# Tabela de dados
angles = [0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180]
probs = [np.cos(np.radians(a/2))**2 for a in angles]

print("\n┌──────────┬───────────┬────────────┐")
print("│ Ângulo θ │ P(passa)  │ P(bloqueado) │")
print("├──────────┼───────────┼────────────┤")
for a, p in zip(angles, probs):
    print(f"│ {a:3d}°     │ {p:6.1%}   │ {1-p:6.1%}     │")
print("└──────────┴───────────┴────────────┘")

# Gráfico
plt.figure(figsize=(10, 6))
plt.plot(angles, probs, 'bo-', markersize=8)
plt.xlabel('Ângulo θ (graus)')
plt.ylabel('P(passa)')
plt.title('Probabilidade de Passagem pelo SGA')
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
plt.show()

---
## 10.7 - One-Time Pad
---

In [ ]:
def xor_bytes(a: bytes, b: bytes) -> bytes:
    """XOR two byte strings."""
    return bytes(x ^ y for x, y in zip(a, b))

def str_to_bin(s: str) -> str:
    """Convert string to binary representation."""
    return ''.join(format(ord(c), '08b') for c in s)

# Exemplo
message = "H"
key = bytes([0b01101010])
message_bytes = message.encode()

cipher = xor_bytes(message_bytes, key)
decrypted = xor_bytes(cipher, key)

print(f"Mensagem: '{message}' = {str_to_bin(message)}")
print(f"Chave:    {format(key[0], '08b')}")
print(f"Cifra:    {format(cipher[0], '08b')}")
print(f"Decifrado: {format(decrypted[0], '08b')} = '{decrypted.decode()}'")

In [ ]:
print("""
QUESTÕES SOBRE ONE-TIME PAD:

Q: Quantas chaves para quebrar 1 letra (8 bits)?
A: 2⁸ = 256 chaves possíveis

Q: E para 5 letras (40 bits)?
A: 2⁴⁰ ≈ 1.1 TRILHÃO de chaves!

Q: Por que é teoricamente inquebrável?
A: 
  1. Cada bit da cifra é completamente ALEATÓRIO
  2. Não há padrões para explorar
  3. TODAS as mensagens são igualmente prováveis
  4. Mesmo com poder computacional INFINITO, é impossível!

Q: Qual a falha prática?
A: A DISTRIBUIÇÃO DE CHAVES:
  - Chave deve ser tão longa quanto a mensagem
  - Deve ser verdadeiramente aleatória
  - Deve ser compartilhada de forma segura
  - NUNCA pode ser reutilizada
  
  Paradoxo: Se há canal seguro para a chave, 
            por que não usar para a mensagem?
  
  SOLUÇÃO: Protocolo BB84 usa mecânica quântica para 
           distribuir chaves de forma segura!
""")

---
## Resumo do Capítulo 10

### Conceitos-Chave:

1. **Estados emaranhados vs estados produto**
2. **Correlações quânticas** persistem em qualquer base
3. **Superposição vs mistura**: distinguíveis por mudança de base
4. **Medição**: probabilidades seguem cos²(θ/2)
5. **One-time pad**: segurança perfeita mas impraticável
6. **BB84**: distribuição de chaves quântica segura